# SILVER_SIA

Produção ambulatorial com tipos corrigidos e descrições resolvidas:
- `PA_CMP` (`AAAAMM`) vira `DT_COMPETENCIA` (1º dia do mês);
- `PA_TPUPS` vira `TIPO_UNIDADE` e `PA_NIVCPL` vira `COMPLEXIDADE` (tabelas oficiais);
- `PA_GESTAO = 270000` (código do estado) indica gestão **ESTADUAL**; o código de um município indica **MUNICIPAL**;
- `PA_UFMUN` é o município do atendimento e `PA_MUNPCN` o de residência do paciente
  (`999999` = residência não informada, característica do BPA consolidado).

**Origem:** `BRONZE_SIA` + `SILVER_DOMINIO`.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO SILVER_SIA
    (DT_COMPETENCIA, COD_CNES, TIPO_UNIDADE, TIPO_GESTAO, COD_IBGE_6_ATEND, COD_IBGE_6_RES,
     COD_PROCEDIMENTO, COMPLEXIDADE, QTD_APRESENTADA, QTD_APROVADA, VL_APRESENTADO, VL_APROVADO)
SELECT TO_DATE(s.PA_CMP || '01' DEFAULT NULL ON CONVERSION ERROR, 'YYYYMMDD'),
       s.PA_CODUNI,
       NVL(t.DESCRICAO, 'NAO INFORMADO'),
       CASE WHEN s.PA_GESTAO IS NULL THEN 'NAO INFORMADO'
            WHEN s.PA_GESTAO LIKE '__0000' THEN 'ESTADUAL'
            ELSE 'MUNICIPAL' END,
       s.PA_UFMUN,
       s.PA_MUNPCN,
       s.PA_PROC_ID,
       NVL(c.DESCRICAO, 'NAO INFORMADO'),
       NVL(s.PA_QTDPRO, 0),
       NVL(s.PA_QTDAPR, 0),
       NVL(s.PA_VALPRO, 0),
       NVL(s.PA_VALAPR, 0)
FROM BRONZE_SIA s
LEFT JOIN SILVER_DOMINIO t ON t.TABELA = 'TIPO_UNIDADE' AND t.CODIGO = s.PA_TPUPS
LEFT JOIN SILVER_DOMINIO c ON c.TABELA = 'COMPLEXIDADE' AND c.CODIGO = s.PA_NIVCPL
"""

In [ ]:
recarregar("SILVER_SIA", SQL, origem="BRONZE_SIA")

In [ ]:
consultar("SELECT * FROM SILVER_SIA FETCH FIRST 10 ROWS ONLY")